In [ ]:
import pandas as pd
import numpy as np
# import os
# os.environ["XLA_FLAGS"] = "--xla_cpu_multi_thread_eigen=true --xla_cpu_multi_thread_eigen_num_threads=4"
import numpyro
numpyro.set_host_device_count(4)
import matplotlib.pyplot as plt

df = pd.read_csv('../source/df.csv')
df

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
def make_fourier_series(t, period, order):
    """
    Args
    ----
    t: array-like, time points at which to evaluate the Fourier series
    period: int, the period of the seasonality
    order: int, the number of Fourier terms to include
    """
    sin_terms = np.array([np.sin(2 * np.pi * i * t / period) for i in range(1, order + 1)])
    cos_terms = np.array([np.cos(2 * np.pi * i * t / period) for i in range(1, order + 1)])
    return np.concatenate((sin_terms, cos_terms), axis=0).transpose(1, 0)

In [ ]:
response = df["sales"].values
# response_norm = response.mean()
response_norm = 1.
y = np.log(response / response_norm)

In [ ]:
x_glb_trend = np.arange(len(y)) / 365.25
x_annual_seas = make_fourier_series(np.arange(len(y)), period=365.25, order=3)
x_weekly_seas = make_fourier_series(np.arange(len(y)), period=7, order=2)
x_seas = np.concatenate((x_annual_seas, x_weekly_seas), axis=1)
print(x_seas.shape)

In [ ]:
lev_sm = 0.001
slp_sm = 0.001
theta = 0.8

In [ ]:
from wunku.models.dlt import (
    run_dlt_model, 
    generate_in_sample_forecast, 
)
from wunku.hyper_tune import (
    generate_forecast_span_samples,
    compute_log_likelihood,
    compute_wbic,
)

In [ ]:
posteriors_dict = run_dlt_model(
    y=y,
    x_glb_trend=x_glb_trend,
    x_seas=x_seas,
    lev_sm=lev_sm,
    slp_sm=slp_sm,
    theta=theta,
)

In [ ]:
yhat_lower, yhat_mid, yhat_upper = generate_in_sample_forecast(
    posteriors_dict=posteriors_dict,
)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(np.arange(len(y)), response, label='Observed', alpha=0.5, color="orange", s=10)
ax.plot(np.arange(len(y)), yhat_mid, label='Forecast', linestyle='--', alpha=0.8, color="dodgerblue")
ax.fill_between(np.arange(len(y)), yhat_lower, yhat_upper, alpha=0.5, label='95% Prediction Interval', color="dodgerblue")
ax.set_title('Damped Local Trend Forecast')

In [ ]:
h=28
yhat_span = generate_forecast_span_samples(posteriors_dict=posteriors_dict, h=h)
loglk = compute_log_likelihood(
    yhat_span=yhat_span,
    sigma=posteriors_dict["sigma"],
    y=y,
)
print(f"loglk shape:", loglk.shape)
wbic = compute_wbic(loglk)
print(f"WBIC: {wbic:.3f}")

Now, let's do a optimization run to find the best hyperparameters for the DLT model.

In [ ]:
h = 28
def run_dlt_model(params):
    lev_sm, slp_sm, theta = params
    print(f"Trying lev_sm={lev_sm:.4f}, slp_sm={slp_sm:.4f}, theta={theta:.4f}")

    kernel = NUTS(dlt_model, init_strategy=init_to_median(num_samples=10))
    mcmc = MCMC(kernel, num_warmup=1000, num_samples=1000, num_chains=4)
    mcmc.run(jax.random.PRNGKey(0), lev_sm=lev_sm, slp_sm=slp_sm, theta=theta)
    posteriors_dict = mcmc.get_samples()
    
    dlt_comp = posteriors_dict["dlt_comp"]
    beta_glb_trend = posteriors_dict["beta_glb_trend"]
    beta_seas = posteriors_dict["beta_seas"]
    sigma = posteriors_dict["sigma"]

    reg_comp = np.sum(x_seas * jnp.expand_dims(beta_seas, -2), axis=-1) + x_glb_trend * jnp.expand_dims(beta_glb_trend, -1)

    dlt_comp_slice = dlt_comp[:, :-(h-1), None]
    reg_comp_slice = slice_trend(reg_comp, h=h)
    yhat_span = dlt_comp_slice + reg_comp_slice
    y_slice = slice_trend_single(y, h=h)

    loglk = compute_log_likelihood(yhat_span, sigma, y_slice)
    wbic = compute_wbic(loglk)
    wbic = float(wbic)

    print("WBIC:", wbic)
    return wbic

# test run_dlt_model()
run_dlt_model((0.001, 0.001, 0.8))

In [ ]:
result.x

In [ ]:
from skopt import gp_minimize
from skopt.space import Real

# Define the hyperparam space
search_space = [
    Real(0.0001, 0.01, prior='log-uniform', name='lev_sm'),
    Real(0.001, 0.1, prior='log-uniform', name='slp_sm'),
    Real(0.8, 0.99, prior='uniform', name='theta'),
]

# Run Bayesian Optimization
result = gp_minimize(
    func=run_dlt_model,  
    dimensions=search_space,
    n_calls=15,                   
    random_state=42
)

In [ ]:
# Print best result
best_lev_sm = result.x[0]
best_slp_sm = result.x[1]
best_theta = result.x[2]

print(f"Best parameters: lev_sm={best_lev_sm:.4f}, slp_sm={best_slp_sm:.4f}, theta={best_theta:.4f}")
print(f"Best WBIC: {result.fun}")

In [ ]:
from skopt.plots import plot_convergence

plot_convergence(result)
plt.show()

In [ ]:
from skopt.plots import plot_evaluations

plot_evaluations(result)
plt.show()


In [ ]:
from skopt.plots import plot_objective

plot_objective(result)
plt.show()
